# Train PatchTST + CVAE on Colab

Run this notebook's kernel connected to a Colab runtime (VS Code: kernel picker top-right -> "Select Another Kernel" -> Google Colab -> pick a GPU runtime).

Run the cells top to bottom. The clone step is idempotent (pulls if already cloned).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os

REPO_URL = "https://github.com/WoodyChang21/ECE1508_GenAI.git"
BRANCH = "PatchTST_OLCV"

if not os.path.isdir("ECE1508_GenAI"):
    !git clone -b {BRANCH} {REPO_URL}
else:
    !cd ECE1508_GenAI && git pull

%cd ECE1508_GenAI

In [ ]:
# torch is preinstalled on Colab; transformers is needed for the HF PatchTSTModel-backed
# train_patchtst.py (src/models/patchtst_hf.py); mplfinance/pyyaml are for evaluate.py/configs.
!pip install -q transformers mplfinance pyyaml

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## Sanity checks (data pipeline tests)

Cheap to run first -- confirms the feature/window logic before committing to a long training run.

In [ ]:
!pip install -q pytest
!python -m pytest steven/tests/ -v

## Train PatchTST (HF PatchTSTModel-backed, real defaults: 20k windows/epoch, 20 epochs, configs/patchtst.yaml)

Now trains `src/models/patchtst_hf.py` instead of the original hand-rolled `src/models/patchtst.py`
-- see `docs/experiments.md` for the comparison. `channel_attention=False` is the config default
(the main approach for now); add `--channel-attention` below to try the mixing variant instead.
Saves to `steven/outputs/patchtst_hf_checkpoint.pt` (separate from the original model's checkpoint).

Drop `--max-epochs`/`--train-windows-per-epoch` overrides below if you want a quick smoke run first
instead of the full config.

In [ ]:
!python steven/src/train_patchtst.py --config steven/configs/patchtst.yaml --device auto

## Train CVAE (real defaults: 20k windows/epoch, 30 epochs, configs/cvae.yaml)

In [ ]:
!python steven/src/train_cvae.py --config steven/configs/cvae.yaml --device auto

## Evaluate both models on the fixed test set

**Not yet updated for the HF PatchTST model** -- `evaluate.py` still assumes the original
`patchtst.py`'s variable-context, padding-mask-aware interface, and the cell below still
points at `patchtst_checkpoint.pt` (the original model's checkpoint, not the new
`patchtst_hf_checkpoint.pt`). Only run this cell if you've separately trained the original
model too; otherwise skip it until evaluate.py is updated for the HF model.

In [ ]:
!python steven/src/evaluate.py \
  --patchtst-checkpoint steven/outputs/patchtst_checkpoint.pt \
  --cvae-checkpoint steven/outputs/cvae_checkpoint.pt \
  --device auto

## Refresh v1.md from this run

Rewrites the Results/backtest tables and sample images in `steven/v1.md` from the metrics.json + sample_plots this run just produced (see `steven/src/update_report.py`). Only the tables/images are rewritten -- surrounding prose (interpretation, caveats) is left as-is; review it by hand if the story changed. This only edits the file in the cloned repo here -- push/download separately if you want to keep it.

In [ ]:
!python steven/src/update_report.py

## Pull results back down

Zips `steven/outputs/` (checkpoints, metrics.json, sample_plots) and downloads it -- or just `git add`/`commit`/`push` from here if you'd rather sync back through the repo.

In [ ]:
!zip -r outputs.zip steven/outputs

try:
    from google.colab import files
    files.download("outputs.zip")
except ImportError:
    print("Not in a Colab frontend session -- outputs.zip is in the working dir, grab it manually.")